# Inland Excess Water (IEW) Monitoring — Stage E: SplitClass Segmentation

Implements **§3.2.1, Algorithm 1** of the paper: an inference-time post-processing step applied to the (self-distilled) Student's softmax output. Where the model is confident (top-1 probability ≥ θ), keep a single class; where it isn't, output a **composite dual-class label** (top-1 + top-2) instead of forcing an arbitrary single answer.

```
for each pixel x_i:
    C_top1 = argmax_k P(x_i, C_k)
    if P(x_i, C_top1) >= theta:        # theta = 0.98 in the paper
        x_i = C_top1                   # confident -> single class
    else:
        C_top2 = argmax_{k != C_top1} P(x_i, C_k)
        x_i = (C_top1, C_top2)         # uncertain -> composite dual-class label
```

A prediction counts as correct if **either** assigned class matches ground truth. With 6 base classes, `C(6,2) = 15` possible composite pairs.

**Pipeline position:** `A → B → C → D. Knowledge Distillation (05_knowledge_distillation.ipynb) → E. SplitClass Segmentation (this notebook) → F. Temporal Gap-Filling`

**Prerequisite:** run `05_knowledge_distillation.ipynb` first — it produces `outputs/checkpoints/rupnet_student_best.pt`. If that file doesn't exist, this notebook falls back to the Stage-C RUPNet (`rupnet_best.pt`) with a printed warning, so it can still run standalone, but the paper applies SplitClass specifically to the **student's** output.

Flow: imports → data pipeline → model → load student (or fallback) → Algorithm 1 implementation → evaluation at θ=0.98 (Table 2 layout + confusion analysis) → θ sweep ablation → visualization of dual-label pixels.

## 1. Imports and Setup

In [ ]:
import os
import sys
import glob
import yaml
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from rasterio.windows import Window
from pathlib import Path
from typing import List, Dict, Tuple

torch.manual_seed(0)
np.random.seed(0)

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")


## 2. Data Pipeline

In [ ]:
# --- Raster I/O helpers -----------------------------------------------------
def read_s2(path: str) -> np.ndarray:
    """Read a Sentinel-2 GeoTIFF -> (11, H, W) float32, scaled to reflectance [0, ~1.7]."""
    with rasterio.open(path) as src:
        data = src.read().astype(np.float32)
        data /= 10000.0  # S2 BOA reflectance is stored scaled by 10000
        return data

def read_s1(path: str) -> np.ndarray:
    """Read a Sentinel-1 GeoTIFF -> (3, H, W) float32: VV, VH, VV/VH ratio (dB)."""
    with rasterio.open(path) as src:
        return src.read().astype(np.float32)

def read_gt(path: str) -> np.ndarray:
    """Read a ground-truth GeoTIFF -> (H, W) int class map."""
    with rasterio.open(path) as src:
        return src.read(1)


def get_scene_files(dataset_root: str, split: str) -> List[Dict[str, str]]:
    """Find and align S1, S2, GT, GT_0_8 files for a split ('Train'/'Test'), keyed off S2."""
    root = Path(dataset_root)
    split_dir = root / "tifs" / split

    s1_dir = split_dir / "S1"
    s2_dir = split_dir / "S2"
    gt_dir = split_dir / "GT"
    gt08_dir = split_dir / "GT_0_8"

    s2_files = sorted(split_dir.joinpath("S2").glob("*.tif"))

    aligned = []
    for s2_file in s2_files:
        name = s2_file.name
        s1_path = s1_dir / name
        gt_path = gt_dir / name
        gt08_path = gt08_dir / name

        if not s1_path.exists():
            raise FileNotFoundError(f"Missing S1 file: {s1_path}")
        if not gt_path.exists():
            raise FileNotFoundError(f"Missing GT file: {gt_path}")
        if not gt08_path.exists():
            raise FileNotFoundError(f"Missing GT_0_8 file: {gt08_path}")

        aligned.append({
            "name": s2_file.stem,
            "S1": str(s1_path),
            "S2": str(s2_file),
            "GT": str(gt_path),
            "GT_0_8": str(gt08_path),
        })
    return aligned

print("Raster I/O helpers ready: read_s2, read_s1, read_gt, get_scene_files")


In [ ]:
# --- PyTorch Dataset ---------------------------------------------------------
# Per-channel statistics computed over the training split.
S2_MEAN = np.array([0.0436, 0.0557, 0.0723, 0.0838, 0.1151, 0.1598, 0.1800, 0.1974, 0.2004, 0.2233, 0.1706], dtype=np.float32)
S2_STD  = np.array([0.0182, 0.0233, 0.0272, 0.0340, 0.0380, 0.0598, 0.0686, 0.0751, 0.0728, 0.0674, 0.0606], dtype=np.float32)

S1_MEAN = np.array([-21.6081, -13.6686, -8.0403], dtype=np.float32)
S1_STD  = np.array([3.4706, 3.0644, 2.5309], dtype=np.float32)


class FloodPatchDataset(Dataset):
    """Sliding-window dataset over aligned S1/S2/GT scenes. Reads windows directly from
    disk via rasterio (or caches full scenes in memory), standardizes per-channel, and
    optionally applies flip/rotation augmentation.
    """

    def __init__(
        self,
        dataset_root: str,
        split: str,
        patch_size: int = 64,
        stride: int = 64,
        augment: bool = False,
        cache_in_memory: bool = False,
        load_s1: bool = True,
    ):
        self.dataset_root = dataset_root
        self.split = split
        self.patch_size = patch_size
        self.stride = stride
        self.augment = augment
        self.cache_in_memory = cache_in_memory
        self.load_s1 = load_s1

        self.scenes = get_scene_files(dataset_root, split)

        self.s2_cache, self.s1_cache, self.gt_cache = {}, {}, {}
        self.patches = []

        print(f"Initializing {split} dataset (cache_in_memory={cache_in_memory}, load_s1={load_s1})...")

        for scene_idx, scene in enumerate(self.scenes):
            with rasterio.open(scene["S2"]) as s2_src:
                H, W = s2_src.height, s2_src.width

            if cache_in_memory:
                with rasterio.open(scene["S2"]) as s2_src:
                    s2_img = s2_src.read().astype(np.float32)
                    s2_img /= 10000.0
                    self.s2_cache[scene_idx] = s2_img
                if load_s1:
                    with rasterio.open(scene["S1"]) as s1_src:
                        self.s1_cache[scene_idx] = s1_src.read().astype(np.float32)
                with rasterio.open(scene["GT_0_8"]) as gt_src:
                    self.gt_cache[scene_idx] = gt_src.read(1).astype(np.int64)
            # else: no persistent file handles are opened here - see __getitem__, which
            # opens+reads+closes per access. This keeps the Dataset free of unpicklable
            # state (required for num_workers > 0, since DataLoader workers get a pickled
            # copy of the Dataset) and avoids GDAL's internal per-handle cache growing
            # unbounded over a long-running, many-epoch loop that reuses the same handle.

            for y in range(0, H - patch_size + 1, stride):
                for x in range(0, W - patch_size + 1, stride):
                    self.patches.append({"scene_idx": scene_idx, "y": y, "x": x})

        print(f"Loaded {split} dataset with {len(self.patches)} total patches from {len(self.scenes)} scenes.")

    def __len__(self) -> int:
        return len(self.patches)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        info = self.patches[idx]
        scene_idx, y, x = info["scene_idx"], info["y"], info["x"]

        if self.cache_in_memory:
            s2_data = self.s2_cache[scene_idx][:, y:y + self.patch_size, x:x + self.patch_size].copy()
            if self.load_s1:
                s1_data = self.s1_cache[scene_idx][:, y:y + self.patch_size, x:x + self.patch_size].copy()
            else:
                s1_data = np.zeros((3, self.patch_size, self.patch_size), dtype=np.float32)
            gt_data = self.gt_cache[scene_idx][y:y + self.patch_size, x:x + self.patch_size].copy()
        else:
            scene = self.scenes[scene_idx]
            window = Window(x, y, self.patch_size, self.patch_size)
            with rasterio.open(scene["S2"]) as s2_src:
                s2_data = s2_src.read(window=window).astype(np.float32)
            s2_data /= 10000.0
            if self.load_s1:
                with rasterio.open(scene["S1"]) as s1_src:
                    s1_data = s1_src.read(window=window).astype(np.float32)
            else:
                s1_data = np.zeros((3, self.patch_size, self.patch_size), dtype=np.float32)
            with rasterio.open(scene["GT_0_8"]) as gt_src:
                gt_data = gt_src.read(1, window=window).astype(np.int64)

        for c in range(11):
            s2_data[c] = (s2_data[c] - S2_MEAN[c]) / S2_STD[c]
        if self.load_s1:
            for c in range(3):
                s1_data[c] = (s1_data[c] - S1_MEAN[c]) / S1_STD[c]

        if self.augment:
            if np.random.rand() > 0.5:
                s2_data = np.flip(s2_data, axis=2).copy()
                if self.load_s1:
                    s1_data = np.flip(s1_data, axis=2).copy()
                gt_data = np.flip(gt_data, axis=1).copy()
            if np.random.rand() > 0.5:
                s2_data = np.flip(s2_data, axis=1).copy()
                if self.load_s1:
                    s1_data = np.flip(s1_data, axis=1).copy()
                gt_data = np.flip(gt_data, axis=0).copy()
            rot_k = np.random.randint(0, 4)
            if rot_k > 0:
                s2_data = np.rot90(s2_data, k=rot_k, axes=(1, 2)).copy()
                if self.load_s1:
                    s1_data = np.rot90(s1_data, k=rot_k, axes=(1, 2)).copy()
                gt_data = np.rot90(gt_data, k=rot_k, axes=(0, 1)).copy()

        return torch.from_numpy(s2_data), torch.from_numpy(s1_data), torch.from_numpy(gt_data)


print("FloodPatchDataset ready")


In [ ]:
def stretch_rgb(rgb: np.ndarray, low: float = 2.0, high: float = 98.0) -> np.ndarray:
    """Per-channel percentile contrast stretch for a true-color composite (see 03_rupnet_backbone.ipynb
    diagnosis note - a fixed divisor washes scenes out or leaves them too dark)."""
    out = np.empty_like(rgb, dtype=np.float32)
    for c in range(rgb.shape[-1]):
        lo, hi = np.percentile(rgb[..., c], [low, high])
        out[..., c] = np.clip((rgb[..., c] - lo) / (hi - lo + 1e-8), 0, 1)
    return out


print("stretch_rgb ready")


## 3. Model Architecture — RUPNet

In [ ]:
class Conv2D(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, padding=1, stride=1, dilation=1, bias=False, act=True):
        super().__init__()
        self.act = act
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size, stride=stride, padding=padding, dilation=dilation, bias=bias),
            nn.BatchNorm2d(out_c),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        if self.act:
            x = self.relu(x)
        return x


class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
        )
        self.shortcut = nn.Sequential()
        if in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1, stride=1, bias=False),
                nn.BatchNorm2d(out_c),
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv(x) + self.shortcut(x))


class DecoderBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.res = ResidualBlock(in_c + out_c, out_c)

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.res(x)


class UpBlock(nn.Module):
    def __init__(self, in_c, out_c, scale):
        super().__init__()
        self.up = nn.Upsample(scale_factor=scale, mode="bilinear", align_corners=True)
        self.res = ResidualBlock(in_c, out_c)

    def forward(self, x):
        return self.res(self.up(x))


class RUPNet(nn.Module):
    """RUPNet (Tomar, Bagci & Jha, 2023) - the paper's winning backbone (Table 1,
    F1 = 81.19%) and its knowledge-distillation teacher/student architecture (~2.05M params
    at in_channels=11; scales down automatically for fewer input channels, e.g. the 3-channel
    Radar_Model)."""

    def __init__(self, in_channels: int = 11, num_classes: int = 6):
        super().__init__()

        self.init_conv = Conv2D(in_channels, 32, kernel_size=3, padding=1)

        self.enc1_1 = ResidualBlock(32, 64)
        self.down1 = nn.MaxPool2d(2)
        self.enc1_2 = ResidualBlock(64, 64)
        self.down2 = nn.MaxPool2d(2)  # scale: H/4

        self.enc2 = ResidualBlock(64, 128)
        self.down3 = nn.MaxPool2d(2)  # scale: H/8

        self.enc3 = ResidualBlock(128, 256)
        self.down4 = nn.MaxPool2d(2)  # scale: H/16

        self.c1 = Conv2D(64, 64, kernel_size=1, padding=0)
        self.c2 = Conv2D(128, 64, kernel_size=1, padding=0)
        self.c3 = Conv2D(256, 64, kernel_size=1, padding=0)

        self.d1 = DecoderBlock(64, 64)
        self.d2 = DecoderBlock(64, 64)

        self.d3 = UpBlock(64, 64, scale=4)
        self.u1 = UpBlock(64, 64, scale=4)
        self.u2 = UpBlock(64, 64, scale=8)
        self.u3 = UpBlock(64, 64, scale=16)

        self.fusion = ResidualBlock(256, 64)
        self.outc = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, inputs):
        x = self.init_conv(inputs)

        e1 = self.enc1_1(x)
        e1 = self.down1(e1)
        e1 = self.enc1_2(e1)
        e1 = self.down2(e1)  # (B, 64, H/4, W/4)

        e2 = self.enc2(e1)
        e2 = self.down3(e2)  # (B, 128, H/8, W/8)

        e3 = self.enc3(e2)
        e3 = self.down4(e3)  # (B, 256, H/16, W/16)

        c1 = self.c1(e1)
        c2 = self.c2(e2)
        c3 = self.c3(e3)

        d1 = self.d1(c3, c2)  # (B, 64, H/8, W/8)
        d2 = self.d2(d1, c1)  # (B, 64, H/4, W/4)

        d3 = self.d3(d2)  # (B, 64, H, W)
        u1 = self.u1(c1)  # (B, 64, H, W)
        u2 = self.u2(c2)  # (B, 64, H, W)
        u3 = self.u3(c3)  # (B, 64, H, W)

        fused = torch.cat([d3, u1, u2, u3], dim=1)  # (B, 256, H, W)
        fused = self.fusion(fused)  # (B, 64, H, W)

        return self.outc(fused)


print("RUPNet ready")


## 4. Load Configuration, Dataset, and the Trained Model
Prefers the Stage-D **student** checkpoint; falls back to the Stage-C teacher-only RUPNet with a warning if Stage D hasn't been run yet.

In [ ]:
with open("configs/data.yaml", "r") as f:
    config = yaml.safe_load(f)
dataset_root = config["dataset_root"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

STUDENT_CKPT = "outputs/checkpoints/rupnet_student_best.pt"
TEACHER_CKPT = "outputs/checkpoints/rupnet_best.pt"

if os.path.exists(STUDENT_CKPT):
    ckpt_path, MODEL_SOURCE = STUDENT_CKPT, "Stage-D Student"
elif os.path.exists(TEACHER_CKPT):
    ckpt_path, MODEL_SOURCE = TEACHER_CKPT, "Stage-C Teacher-only (fallback - run 05_knowledge_distillation.ipynb for the paper-faithful setup)"
else:
    raise FileNotFoundError("Neither rupnet_student_best.pt nor rupnet_best.pt found - run 03_rupnet_backbone.ipynb (and ideally 05_knowledge_distillation.ipynb) first.")

model = RUPNet(in_channels=11, num_classes=6).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded {MODEL_SOURCE} from {ckpt_path} (epoch {ckpt['epoch']}, val_loss {ckpt['val_loss']:.4f})")

class_names = [
    "Open water (2)", "Soil seriously affected (3)", "Soil moderately affected (4)",
    "Soil slightly affected (5)", "Vegetation affected (6)", "Not affected (7)",
]

eval_dataset = FloodPatchDataset(
    dataset_root=dataset_root, split="Test", patch_size=config["patch_size"],
    stride=config["patch_size"], augment=False, cache_in_memory=False, load_s1=False,
)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False, num_workers=0)


## 5. Algorithm 1 — SplitClass Segmentation

In [ ]:
def splitclass_predict(logits: torch.Tensor, theta: float):
    """Algorithm 1. Returns (top1, top2, is_dual) as numpy arrays of shape (B, H, W):
      - top1:   argmax class index
      - top2:   second-best class index (meaningless where is_dual is False, but always computed)
      - is_dual: boolean mask, True where top1 confidence < theta (dual-label pixel)
    """
    probs = F.softmax(logits, dim=1)  # (B, C, H, W)
    top2_probs, top2_idx = torch.topk(probs, k=2, dim=1)  # (B, 2, H, W), sorted descending
    top1 = top2_idx[:, 0]
    top2 = top2_idx[:, 1]
    top1_conf = top2_probs[:, 0]
    is_dual = top1_conf < theta
    return top1.cpu().numpy(), top2.cpu().numpy(), is_dual.cpu().numpy()


def splitclass_correct(top1, top2, is_dual, gt, valid_mask):
    """'Correct' if EITHER assigned class matches GT (paper's evaluation rule).
    For confident (non-dual) pixels this reduces to plain top-1 accuracy."""
    match_top1 = (top1 == gt)
    match_top2 = is_dual & (top2 == gt)
    return (match_top1 | match_top2) & valid_mask


print("Algorithm 1 (splitclass_predict, splitclass_correct) ready")


## 6. Evaluation at θ = 0.98 (paper's default threshold)
Per-class Pr/Rc/F1 using the "either class matches" rule, overall dual-label pixel rate, and confusion analysis for the specific class pairs the paper flags.

In [ ]:
THETA = 0.98

tp = np.zeros(6, dtype=np.int64)
fp = np.zeros(6, dtype=np.int64)
fn = np.zeros(6, dtype=np.int64)
class_pixel_counts = np.zeros(6, dtype=np.int64)

total_pixels = 0
dual_pixels = 0
pair_counts = {}  # (min_class, max_class) -> count, for dual-labeled pixels only

model.eval()
with torch.no_grad():
    for s2_patches, _, gt_patches in eval_loader:
        s2_patches = s2_patches.to(device)
        gt_np = gt_patches.numpy()
        logits = model(s2_patches)
        top1, top2, is_dual = splitclass_predict(logits, THETA)

        valid_mask = gt_np < 6
        total_pixels += valid_mask.sum()
        dual_pixels += (is_dual & valid_mask).sum()

        correct = splitclass_correct(top1, top2, is_dual, gt_np, valid_mask)

        for c in range(6):
            gt_c = (gt_np == c) & valid_mask
            # a class-c GT pixel is a "hit" for precision/recall purposes if it is correctly
            # covered (either assigned label == c); predicted-as-c set = pixels where c is
            # among the assigned label(s) AND that pixel counts as correct for its own GT class,
            # OR pixels where c is assigned but GT differs (false positive for c).
            pred_has_c = (top1 == c) | (is_dual & (top2 == c))
            tp[c] += np.sum(pred_has_c & gt_c)
            fp[c] += np.sum(pred_has_c & ~gt_c & valid_mask)
            fn[c] += np.sum(~pred_has_c & gt_c)
            class_pixel_counts[c] += np.sum(gt_c)

        # confusion pairs among dual-labeled pixels
        b_idx, h_idx, w_idx = np.where(is_dual & valid_mask)
        for b, h, w in zip(b_idx, h_idx, w_idx):
            pair = tuple(sorted((int(top1[b, h, w]), int(top2[b, h, w]))))
            pair_counts[pair] = pair_counts.get(pair, 0) + 1

pr = tp / (tp + fp + 1e-10)
rc = tp / (tp + fn + 1e-10)
f1 = 2 * pr * rc / (pr + rc + 1e-10)
w = class_pixel_counts / class_pixel_counts.sum()

print("\n" + "=" * 70)
print(f"{'Class':<32}{'Pr (%)':<10}{'Rc (%)':<10}{'F1 (%)':<10}")
print("=" * 70)
for i, name in enumerate(class_names):
    print(f"{name:<32}{pr[i]*100:<10.2f}{rc[i]*100:<10.2f}{f1[i]*100:<10.2f}")
print("=" * 70)
print(f"{'ALL (pixel-weighted)':<32}{np.sum(pr*w)*100:<10.2f}{np.sum(rc*w)*100:<10.2f}{np.sum(f1*w)*100:<10.2f}")
print("=" * 70)
print(f"\nDual-labeled pixels: {dual_pixels:,} / {total_pixels:,} ({100*dual_pixels/total_pixels:.2f}%)")
print("Paper reference at theta=0.98: 6.33% dual-labeled; SplitClass All F1 = 0.99")

print("\nTop confusion pairs among dual-labeled pixels:")
sorted_pairs = sorted(pair_counts.items(), key=lambda kv: kv[1], reverse=True)
for (a, b), cnt in sorted_pairs[:10]:
    pct = 100 * cnt / max(dual_pixels, 1)
    print(f"  {class_names[a]:<30} <-> {class_names[b]:<30} {cnt:>8,} ({pct:.2f}% of dual-labels)")

# Paper specifically flags these two pairs as noteworthy - report them explicitly if present
water_notaffected = pair_counts.get(tuple(sorted((0, 5))), 0)
veg_notaffected = pair_counts.get(tuple(sorted((4, 5))), 0)
print(f"\nWater <-> Not-affected pairs: {100*water_notaffected/max(dual_pixels,1):.2f}% of dual-labels (paper: 0.13% of ALL pixels)")
print(f"Vegetation-affected <-> Not-affected pairs: {100*veg_notaffected/max(dual_pixels,1):.2f}% of dual-labels (paper: 2.33% of ALL pixels, paper's largest confusion pair)")


## 7. Threshold Sweep (ablation beyond the paper)
The paper reports only θ=0.98. Sweeping θ shows the F1-vs-dual-label-rate tradeoff explicitly.

In [ ]:
THETA_SWEEP = [0.5, 0.7, 0.9, 0.95, 0.98, 0.99]
sweep_results = []

for theta in THETA_SWEEP:
    tp_s = np.zeros(6, dtype=np.int64); fp_s = np.zeros(6, dtype=np.int64); fn_s = np.zeros(6, dtype=np.int64)
    cpc_s = np.zeros(6, dtype=np.int64)
    total_s = dual_s = 0

    with torch.no_grad():
        for s2_patches, _, gt_patches in eval_loader:
            s2_patches = s2_patches.to(device)
            gt_np = gt_patches.numpy()
            logits = model(s2_patches)
            top1, top2, is_dual = splitclass_predict(logits, theta)
            valid_mask = gt_np < 6
            total_s += valid_mask.sum()
            dual_s += (is_dual & valid_mask).sum()
            for c in range(6):
                gt_c = (gt_np == c) & valid_mask
                pred_has_c = (top1 == c) | (is_dual & (top2 == c))
                tp_s[c] += np.sum(pred_has_c & gt_c)
                fp_s[c] += np.sum(pred_has_c & ~gt_c & valid_mask)
                fn_s[c] += np.sum(~pred_has_c & gt_c)
                cpc_s[c] += np.sum(gt_c)

    pr_s = tp_s / (tp_s + fp_s + 1e-10)
    rc_s = tp_s / (tp_s + fn_s + 1e-10)
    f1_s = 2 * pr_s * rc_s / (pr_s + rc_s + 1e-10)
    w_s = cpc_s / cpc_s.sum()
    sweep_results.append({
        "theta": theta, "f1_all": float(np.sum(f1_s * w_s)),
        "dual_pct": 100 * dual_s / total_s, "f1_per_class": f1_s,
    })
    print(f"theta={theta:<6} F1(All)={np.sum(f1_s*w_s)*100:6.2f}%  dual-labeled={100*dual_s/total_s:5.2f}%")

fig, ax1 = plt.subplots(figsize=(9, 5))
thetas = [r["theta"] for r in sweep_results]
ax1.plot(thetas, [r["f1_all"] * 100 for r in sweep_results], "o-", color="tab:blue", label="F1 (All, %)")
ax1.set_xlabel("theta")
ax1.set_ylabel("F1 (%)", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(thetas, [r["dual_pct"] for r in sweep_results], "s--", color="tab:orange", label="Dual-labeled (%)")
ax2.set_ylabel("Dual-labeled pixels (%)", color="tab:orange")
plt.title("SplitClass: F1 vs. Dual-Label Rate across theta")
fig.tight_layout()
plt.show()


## 8. Visualizing Dual-Class Assignment
Patches chosen by GT class diversity. The third panel highlights dual-labeled pixels (white) over the top-1 prediction.

In [ ]:
def patch_class_diversity(idx):
    _, _, gt = eval_dataset[idx]
    gt_np = gt.numpy()
    valid = gt_np[gt_np < 6]
    return len(np.unique(valid)) if valid.size else 0

scan_indices = range(0, len(eval_dataset), 37)
diversity_scores = sorted(((idx, patch_class_diversity(idx)) for idx in scan_indices), key=lambda t: t[1], reverse=True)
chosen = [idx for idx, _ in diversity_scores[:3]]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i, patch_idx in enumerate(chosen):
    s2, _, gt = eval_dataset[patch_idx]
    s2_in = s2.unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(s2_in)
        top1, top2, is_dual = splitclass_predict(logits, THETA)
    top1, is_dual = top1[0], is_dual[0]

    s2_np = s2.numpy()
    rgb = np.stack([s2_np[3], s2_np[2], s2_np[1]], axis=-1)
    rgb = rgb * S2_STD[[3, 2, 1]] + S2_MEAN[[3, 2, 1]]
    rgb = stretch_rgb(rgb)

    dual_overlay = np.ma.masked_where(~is_dual, is_dual.astype(float))

    axes[i, 0].imshow(rgb); axes[i, 0].set_title("Input RGB"); axes[i, 0].axis("off")
    axes[i, 1].imshow(gt.numpy(), cmap="tab10", vmin=0, vmax=5); axes[i, 1].set_title("GT Mask"); axes[i, 1].axis("off")
    axes[i, 2].imshow(top1, cmap="tab10", vmin=0, vmax=5); axes[i, 2].set_title(f"Top-1 Prediction ({MODEL_SOURCE.split(chr(32))[0]})"); axes[i, 2].axis("off")
    axes[i, 3].imshow(top1, cmap="tab10", vmin=0, vmax=5, alpha=0.5)
    axes[i, 3].imshow(dual_overlay, cmap="gray_r", vmin=0, vmax=1, alpha=0.9)
    axes[i, 3].set_title(f"Dual-labeled pixels (white), theta={THETA}")
    axes[i, 3].axis("off")

plt.tight_layout()
plt.show()


## 9. Summary — fill in after running

- **F1 (All) at theta=0.98:** ___  (paper: 0.99)
- **Dual-labeled pixel rate at theta=0.98:** ___  (paper: 6.33%)
- **Largest per-class F1 gain vs. plain top-1:** ___ (paper: classes 3-6, the ambiguous soil/vegetation severity gradations)
- **theta sweep takeaway:** ___ (expected: F1 rises and dual-label rate rises together as theta increases toward 1.0 - report where the tradeoff stops being worth it for your data)